# 1. Imports 

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
import optuna
import mlflow
import mlflow.pytorch
from mlflow.tracking import MlflowClient

# 2. Configuration

In [ ]:
sys.path.append("../src")
from preprocessing import (
    get_dataloaders, get_device, denormalize,
    WasteDataset, CLASSES, NUM_CLASSES, IDX_TO_CLASS,
)

device = get_device()
print(f"Device  : {device}")
print(f"Classes : {NUM_CLASSES} — {CLASSES}")

# 3. MLflow + load data

In [ ]:
MLFLOW_URI       = "http://localhost:5000"
EXPERIMENT_NAME  = "sortify-scratch"
REGISTERED_MODEL = "sortify-cnn-scratch"
SPLITS_PATH      = "../data/splits/splits.json"

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

with open(SPLITS_PATH) as f:
    splits = json.load(f)

class_weights = WasteDataset(splits["train"]).get_class_weights().to(device)

print(f"Train : {len(splits['train'])} | Val : {len(splits['val'])} | Test : {len(splits['test'])}")
print(f"Class weights : {[round(w, 3) for w in class_weights.tolist()]}")

# 4. CNN Architecture 

In [ ]:
class ConvBlock(nn.Module):
    """Conv -> BN -> ReLU -> Conv -> BN -> ReLU -> MaxPool optional."""

    def __init__(self, in_ch: int, out_ch: int, pool: bool = True) -> None:
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        ]
        if pool:
            layers.append(nn.MaxPool2d(2, 2))
        self.block = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class WasteCNNScratch(nn.Module):
    """
    CNN from scratch 
    Input  : (B, 3, 224, 224)
    Output : (B, NUM_CLASSES)
    """

    def __init__(self, num_classes: int = NUM_CLASSES, dropout: float = 0.4) -> None:
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3,   32),   # -> 112x112
            ConvBlock(32,  64),   # ->  56x56
            ConvBlock(64,  128),  # ->  28x28
            ConvBlock(128, 256),  # ->  14x14
            ConvBlock(256, 512),  # ->   7x7
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(512, 256), nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))


# Verification
_m = WasteCNNScratch().to(device)
print(f"Output shape    : {_m(torch.randn(2, 3, 224, 224).to(device)).shape}")
print(f"Params totaux   : {sum(p.numel() for p in _m.parameters()):,}")
del _m

# 5. Optuna — Hyperparameter tuning
Each trial = 1 MLflow run. Optuna tests 20 combinations and prunes out the bad trials early on.

In [ ]:
N_TRIALS      = 20
EPOCHS_OPTUNA = 10


def objective(trial: optuna.Trial) -> float:
    lr             = trial.suggest_float("lr",           1e-4, 1e-2, log=True)
    dropout        = trial.suggest_float("dropout",      0.2, 0.6)
    batch_size     = trial.suggest_categorical("batch_size", [16, 32, 64])
    weight_decay   = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "AdamW", "SGD"])

    train_loader, val_loader, _ = get_dataloaders(
        SPLITS_PATH, batch_size=batch_size, pretrained=False
    )
    model     = WasteCNNScratch(dropout=dropout).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    opt       = (
        optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        if optimizer_name == "Adam" else
        optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        if optimizer_name == "AdamW" else
        optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    )
    scheduler     = ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=2)
    best_val_loss = float("inf")
    no_improve    = 0

    with mlflow.start_run(run_name=f"trial_{trial.number}"):
        mlflow.log_params({
            "lr": lr, "dropout": dropout, "batch_size": batch_size,
            "weight_decay": weight_decay, "optimizer": optimizer_name,
            "phase": "optuna",
        })

        for epoch in range(1, EPOCHS_OPTUNA + 1):
            # train
            model.train()
            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)
                opt.zero_grad()
                loss = criterion(model(images), labels)
                loss.backward()
                opt.step()

            # val
            model.eval()
            val_loss, correct, total = 0.0, 0, 0
            with torch.no_grad():
                for images, labels in val_loader:
                    images, labels = images.to(device), labels.to(device)
                    outputs = model(images)
                    val_loss  += criterion(outputs, labels).item() * images.size(0)
                    correct   += (outputs.argmax(1) == labels).sum().item()
                    total     += images.size(0)
            val_loss /= total
            val_acc   = correct / total

            scheduler.step(val_loss)
            mlflow.log_metrics({"val_loss": val_loss, "val_acc": val_acc}, step=epoch)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                no_improve    = 0
            else:
                no_improve += 1

            trial.report(val_loss, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

            if no_improve >= 3:
                break

        mlflow.log_metric("best_val_loss", best_val_loss)

    return best_val_loss


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3),
    study_name="sortify-scratch-optuna",
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best = study.best_params
print("\nMeilleurs hyperparametres :")
for k, v in best.items():
    print(f"  {k:<15} : {v}")
print(f"Meilleur val_loss : {study.best_value:.4f}")

In [ ]:
# Visualization Optuna
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

trial_values = [t.value for t in study.trials if t.value is not None]
axes[0].plot(trial_values, marker="o", color="#7F77DD", markersize=5, linewidth=1)
axes[0].axhline(study.best_value, color="#D85A30", linestyle="--", label=f"Best: {study.best_value:.4f}")
axes[0].set_title("Val loss par trial")
axes[0].set_xlabel("Trial")
axes[0].legend()
axes[0].spines[["top", "right"]].set_visible(False)

importances = optuna.importance.get_param_importances(study)
axes[1].barh(list(importances.keys()), list(importances.values()), color="#5DCAA5")
axes[1].set_title("Importance des hyperparametres")
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("../saved/scratch_optuna.png", dpi=150)
plt.show()

# 6. Final training with the best hyperparameters

In [ ]:
EPOCHS_FINAL  = 30
PATIENCE      = 5
SAVED_DIR     = Path("../saved")
SAVED_DIR.mkdir(parents=True, exist_ok=True)

train_loader, val_loader, test_loader = get_dataloaders(
    SPLITS_PATH, batch_size=best["batch_size"], pretrained=False
)
model     = WasteCNNScratch(dropout=best["dropout"]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = (
    optim.Adam(model.parameters(), lr=best["lr"], weight_decay=best["weight_decay"])
    if best["optimizer"] == "Adam" else
    optim.AdamW(model.parameters(), lr=best["lr"], weight_decay=best["weight_decay"])
    if best["optimizer"] == "AdamW" else
    optim.SGD(model.parameters(), lr=best["lr"], momentum=0.9, weight_decay=best["weight_decay"])
)
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3, min_lr=1e-6)

history       = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_loss = float("inf")
no_improve    = 0

with mlflow.start_run(run_name="final_training") as final_run:
    mlflow.log_params({**best, "epochs": EPOCHS_FINAL, "phase": "final", "architecture": "WasteCNNScratch"})

    print(f"{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>10} {'Train Acc':>10} {'Val Acc':>9}")
    print("-" * 55)

    for epoch in range(1, EPOCHS_FINAL + 1):
        # ── train
        model.train()
        t_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            t_loss  += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total   += images.size(0)
        train_loss, train_acc = t_loss / total, correct / total

        # ── val
        model.eval()
        v_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                v_loss  += criterion(outputs, labels).item() * images.size(0)
                correct += (outputs.argmax(1) == labels).sum().item()
                total   += images.size(0)
        val_loss, val_acc = v_loss / total, correct / total

        scheduler.step(val_loss)
        for k, v in zip(
            ["train_loss", "val_loss", "train_acc", "val_acc"],
            [train_loss, val_loss, train_acc, val_acc]
        ):
            history[k].append(v)
        mlflow.log_metrics({"train_loss": train_loss, "val_loss": val_loss,
                            "train_acc": train_acc, "val_acc": val_acc}, step=epoch)

        print(f"{epoch:>6} {train_loss:>12.4f} {val_loss:>10.4f} {train_acc:>10.4f} {val_acc:>9.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            no_improve    = 0
            torch.save({"epoch": epoch, "model_state": model.state_dict(),
                        "val_loss": val_loss, "val_acc": val_acc,
                        "classes": CLASSES, "hyperparams": best},
                       SAVED_DIR / "scratch_best.pt")
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f"Early stopping a l'epoch {epoch}")
                break

    # ── test set
    ckpt = torch.load(SAVED_DIR / "scratch_best.pt", map_location=device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            all_preds.extend(model(images.to(device)).argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())
    all_preds, all_labels = np.array(all_preds), np.array(all_labels)

    test_acc = float((all_preds == all_labels).mean())
    test_f1  = float(f1_score(all_labels, all_preds, average="macro"))

    mlflow.log_metrics({"test_acc": test_acc, "test_f1_macro": test_f1, "best_val_loss": best_val_loss})

    # ── Model Registry
    mlflow.pytorch.log_model(
        model,
        artifact_path="model",
        registered_model_name=REGISTERED_MODEL,
    )

    print(f"\nTest accuracy  : {test_acc:.4f}")
    print(f"Test F1 macro  : {test_f1:.4f}")
    print(f"Run ID         : {final_run.info.run_id}")

# 7. Learning curves

In [ ]:
x = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, title in zip(axes, ["loss", "acc"], ["Loss", "Accuracy"]):
    ax.plot(x, history[f"train_{metric}"], label="train", color="#5DCAA5")
    ax.plot(x, history[f"val_{metric}"],   label="val",   color="#D85A30")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.spines[["top", "right"]].set_visible(False)

plt.suptitle("CNN Scratch — Courbes d'apprentissage", fontsize=13)
plt.tight_layout()
plt.savefig(SAVED_DIR / "scratch_learning_curves.png", dpi=150)
plt.show()

# 8. Confusion matrix + classification report

In [ ]:
print(classification_report(all_labels, all_preds, target_names=CLASSES, digits=3))

cm_norm = confusion_matrix(all_labels, all_preds).astype(float)
cm_norm /= cm_norm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm_norm, annot=True, fmt=".2f",
            xticklabels=CLASSES, yticklabels=CLASSES,
            cmap="YlOrRd", ax=ax, linewidths=0.5)
ax.set_xlabel("Predit", fontsize=12)
ax.set_ylabel("Reel", fontsize=12)
ax.set_title("Matrice de confusion normalisee — CNN Scratch", fontsize=13, pad=15)
plt.tight_layout()
plt.savefig(SAVED_DIR / "scratch_confusion_matrix.png", dpi=150)
plt.show()

# 9. Save Model Registry

In [ ]:
client   = MlflowClient(MLFLOW_URI)
versions = client.get_latest_versions(REGISTERED_MODEL)
last_ver = versions[-1].version

# Automatically promote to Staging if test_acc > 0.70
if test_acc >= 0.70:
    client.transition_model_version_stage(
        name=REGISTERED_MODEL, version=last_ver, stage="Staging"
    )
    print(f"Modele v{last_ver} promu en Staging (test_acc={test_acc:.4f})")
else:
    print(f"Modele v{last_ver} reste en None — test_acc={test_acc:.4f} < 0.70")

print(f"\nRecapitulatif final :")
print(f"  Test accuracy  : {test_acc:.4f}")
print(f"  Test F1 macro  : {test_f1:.4f}")
print(f"  Best val loss  : {best_val_loss:.4f}")
print(f"  Registered     : {REGISTERED_MODEL} v{last_ver}")